# 学习助手（第 1 天）—— PDF 学习材料摘要

## 练习目标（理念）

做一个小工具：把 **PDF 学习材料** 交给模型，生成适合复习用的摘要。

- **输入**：PDF 文件路径（当前仅支持 PDF；作者注明后续会扩展到更多格式）
- **输出**：可读的 Markdown 摘要，方便安排学习时段
- **技术点**：用 OpenAI **Files API** 上传文件，再用 **Responses API**（`responses.create`）把 `file_id` + 文本指令一并交给模型

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量与密钥 | `.env` + `OPENAI_API_KEY`，启动前做格式校验 |
| OpenAI 客户端 | `OpenAI()` |
| 文件作为上下文 | `openai.files.create(..., purpose="user_data")` |
| 新一代 Responses API | `openai.responses.create` + `input_file` / `input_text` |

## 怎么跑

1. 准备 `.env`，写入有效的 `OPENAI_API_KEY`（通常以 `sk-` 开头）
2. 确认示例 PDF 路径与最后一格参数一致（仓库里文件在本目录旁）
3. 从上到下运行：导入 → 校验 Key → 定义 `process` / `summarize` → 调用 `summarize(...)`


In [25]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在笔记本里用 Markdown 漂亮地显示摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：后面要用 Files API + Responses API
from openai import OpenAI

# 若本格报错：请先去课程的 troubleshooting 笔记本排查环境 / 依赖问题


In [26]:
# ========== 环境校验：确认 OPENAI_API_KEY 可用 ==========

# override=True：.env 中的值覆盖已有环境变量，避免本机残留旧 Key
load_dotenv(override=True)
# 从环境读取 API Key（名字必须是 OPENAI_API_KEY，与官方 SDK 默认一致）
api_key = os.getenv("OPENAI_API_KEY")

# 分三层校验：缺失 / 格式不对 / 空壳 Key；错误文案保持英文（影响行为的字符串不翻译）
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not found. Please set it in your .env file.")
elif not api_key.startswith("sk-"):
    raise ValueError("Invalid API key format. Please ensure your OPENAI_API_KEY starts with 'sk-'.")
elif api_key.strip() == "sk-":
    raise ValueError("API key is empty. Please provide a valid API key in your .env file.")
else:
    # 校验通过：打印成功提示（保持原打印字符串，避免改可观察行为）
    print("API key loaded successfully")


API key loaded successfully


In [27]:
# ========== 核心：上传 PDF + Responses API 生成摘要 ==========

# 创建 OpenAI 客户端（默认从环境变量读 OPENAI_API_KEY）
openai = OpenAI()

def process(document):
    # files.create：以二进制打开本地 PDF，上传到 OpenAI；purpose="user_data" 表示供后续模型输入使用
    f = openai.files.create(
        file=open(document, "rb"),
        purpose="user_data"
    )

    # Newest version：用 Responses API（不是旧的 chat.completions）把文件 + 文本指令一起提交
    response = openai.responses.create(
        # 模型 id 保持原样（影响计费与能力）；不要擅自改成别的模型名
        model = "gpt-5-nano",
        input = [{
            "role": "user",
            "content": [
                # input_file：引用刚上传文件的 file_id
                {"type": "input_file", "file_id": f.id},
                # input_text：用户指令（prompt 保留英文，改译会改变模型行为）
                {"type": "input_text", "text": "Help me summarize this document for my study sessions."}
            ]
        }]
    )
    # output_text：Responses API 提供的便捷属性，直接取模型生成的纯文本
    return response.output_text


In [28]:
# ========== 展示层：调用 process，并把结果渲染成 Markdown ==========

def summarize(document):
    # 先跑摘要流水线（上传文件 → Responses → 文本）
    summary = process(document)
    # 标题行（展示文案保持原样）
    display(Markdown(f"### Summary"))
    # 把模型返回的摘要正文以 Markdown 显示在笔记本里
    display(Markdown(summary))


In [32]:
# ========== 入口：对示例 PDF 跑一遍学习摘要 ==========

# 路径保持原样（相对仓库布局）；若你本地文件位置不同，只改这个字符串参数
# 前提：上一格已定义 summarize；.env 里 Key 有效；该 PDF 存在
summarize("studyHelperDay1/random_study_document.pdf")


### Summary

Here’s a concise study-friendly summary of the document, with quick prompts you can use for review.

High-level overview
- A test document designed to exercise PDF processing: chunking, embedding, retrieval, and cross-page summarization.
- Content spans three sections: Natural Language Processing (NLP), Machine Learning (ML), and Operating Systems (OS).
- Each section includes a short intro, a random number, a repeated filler sentence, and a bullet list of “Key concept” IDs related to that section.
- An additional note confirms the document is multi-page and intended to test cross-page handling.

Section-by-section summary
- Section 1: Natural Language Processing
  - Purpose: Example content for testing PDF processing pipelines.
  - Random number mentioned: 5493.
  - Filler sentence: “The quick brown fox jumps over the lazy dog.”
  - Key concepts listed: 39, 48, 38, 4 (all related to NLP).

- Section 2: Machine Learning
  - Purpose: Example content for testing PDF processing pipelines.
  - Random number mentioned: 5774.
  - Filler sentence: “The quick brown fox jumps over the lazy dog.”
  - Key concepts listed: 38, 2, 39, 17 (all related to ML).

- Section 3: Operating Systems
  - Purpose: Example content for testing PDF processing pipelines.
  - Random number mentioned: 9600.
  - Filler sentence: “The quick brown fox jumps over the lazy dog.”
  - Key concepts listed: 26, 42, 17, 13 (all related to OS).

Additional notes
- The document explicitly notes that a second page is included to test multi-page handling.
- It emphasizes that a PDF study-helper system should chunk, embed, retrieve, and summarize content across page boundaries without losing context.

Study prompts and quick actions
- Identify and memorize the section topics: NLP, ML, OS.
- List the key concept IDs per section:
  - NLP: 39, 48, 38, 4
  - ML: 38, 2, 39, 17
  - OS: 26, 42, 17, 13
- Note the repeated filler sentence as a non-substantive placeholder used for testing.
- Practice a cross-page summary: write a short synthesis that connects the NLP/ML/OS sections as a single document, then test chunking across the page boundary.
- Quick quiz ideas:
  - Which section mentions random number 5774?
  - How many key concepts are listed per section?
  - What is the universal filler sentence used in all sections?
  - What is the purpose of the additional notes?